In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# AMD energy attribution

In [2]:
# AMD (uProf): attribute per-task energy by summing package power (W, ~1 Hz)
# within each task's [start, complete] window. uProf logs local time; the
# Nextflow trace logs epoch-ms UTC, so timestamps are aligned to local seconds.
def process_uprof_run(timechart_path, trace_path, output_path, source_dataset, source_workflow, node, tz_offset_h0urs=1,trace_sep=","):

    with open(timechart_path) as f:
        lines = f.readlines()

    hdr = next(i for i,l in enumerate(lines) if l.startswith("RecordId"))
    uprof_data = pd.read_csv(timechart_path,skiprows=hdr, encoding="latin-1")
    uprof_data.columns = [c.strip() for c in uprof_data.columns]
    uprof_data = uprof_data.dropna(subset=["Timestamp"])

    def uprof_to_sec(t):
        h,m,s,ms = map(int,str(t).split(":"))
        return h*3600 + m*60 + s + ms/1000
    uprof_data["t_sec"] = uprof_data["Timestamp"].apply(uprof_to_sec)
    uprof_data["pkg_w"] = uprof_data["socket0-package-power"].astype(float)

    tr = pd.read_csv(trace_path,
                 sep=",", engine="python", quoting=3)
    tr.columns = [c.strip().strip('"') for c in tr.columns]

    for c in ["start", "complete", "realtime", "peak_rss", "rchar", "cpus"]:
        tr[c] = pd.to_numeric(tr[c], errors="coerce")

    # Trace timestamps are epoch ms in UTC; uProf logged local (BST = UTC+1).
    # Convert both start and complete to local seconds-since-midnight.
    for c in ["start", "complete"]:
        dt = pd.to_datetime(tr[c], unit="ms") + pd.Timedelta(hours=1)   # UTC -> BST
        tr[c + "_sec"] = dt.dt.hour*3600 + dt.dt.minute*60 + dt.dt.second + dt.dt.microsecond/1e6

    tr["task_type"] = tr["process"].str.split(":").str[-1].str.upper()

    # ---------- 3. Verify uProf covers the workflow ----------
    covers = (uprof_data["t_sec"].min() <= tr["start_sec"].min()) and (uprof_data["t_sec"].max() >= tr["complete_sec"].max())
    print(f"uProf covers workflow: {covers}")
    if not covers:
        print("  WARNING: power capture does not fully span the workflow — attribution will be incomplete")

    # ---------- 4. Per-task energy: sum power within each task's window ----------
    # Each uProf sample is ~1 second, so summing watts within [start, complete] ≈ joules.
    def task_energy(row):
        window = uprof_data[(uprof_data["t_sec"] >= row["start_sec"]) & (uprof_data["t_sec"] <= row["complete_sec"])]
        if len(window) == 0:
            return np.nan            # task fell between samples (sub-second)
        return window["pkg_w"].sum()

    tr["measured_energy_j"] = tr.apply(task_energy, axis=1)

    got = tr["measured_energy_j"].notna().sum()
    print(f"Per-task energy attributed: {got} / {len(tr)} tasks "
        f"({len(tr)-got} too short, fell between samples)")


    # ---------- 5. Total workflow energy ----------
    wf = uprof_data[(uprof_data["t_sec"] >= tr["start_sec"].min()) & (uprof_data["t_sec"] <= tr["complete_sec"].max())]
    total = wf["pkg_w"].sum()
    print(f"Total workflow energy: {total:,.0f} J ({total/3600:.2f} Wh), "
        f"mean {wf['pkg_w'].mean():.1f} W over {(tr['complete_sec'].max()-tr['start_sec'].min())/60:.1f} min")

    # --- Labels ---
    tr["source_dataset"] = source_dataset
    tr["source_workflow"] = source_workflow
    tr["node"]  = node
    tr["runtime_s"] = tr["realtime"] 
    tr["peak_mem"] = tr["peak_rss"]
    tr["energy_j"] = tr["measured_energy_j"]
    # ---------- 6. Sanity check: input size vs measured energy ----------
    e = tr["energy_j"].notna().sum()
    r = tr["runtime_s"].notna().sum()
    m = tr["peak_mem"].notna().sum()
    print(f"  Attributed -> energy: {e}, runtime: {r}, memory: {m} (of {len(tr)})")

    v = tr.dropna(subset=["energy_j", "rchar"])
    v = v[v["energy_j"] > 0]
    if len(v) > 2:
        rho, _ = spearmanr(v["rchar"], v["energy_j"])
        print(f"  Sanity (rchar vs energy): Spearman {rho:.3f} (n={len(v)})")


    tr.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    return tr

In [3]:
ryzen5_atacseq = process_uprof_run(timechart_path="../raw/AMD_Ryzen5/atacseq/timechart.csv",
                                trace_path="../raw/AMD_Ryzen5/atacseq/trace.csv",
                                output_path="../Processed_data/Experiments/AMD_Ryzen5/atacseq/measured/atacseq_ryzen5_measured.csv",
                                source_dataset="ryzen5_laptop", source_workflow="atacseq",
                                node="amd_ryzen_5", trace_sep = ",")



uProf covers workflow: True
Per-task energy attributed: 243 / 265 tasks (22 too short, fell between samples)
Total workflow energy: 35,724 J (9.92 Wh), mean 22.6 W over 26.5 min
  Attributed -> energy: 243, runtime: 265, memory: 265 (of 265)
  Sanity (rchar vs energy): Spearman 0.790 (n=243)
Saved: ../Processed_data/Experiments/AMD_Ryzen5/atacseq/measured/atacseq_ryzen5_measured.csv


In [4]:
ryzen5_chipseq = process_uprof_run(timechart_path="../raw/AMD_Ryzen5/chipseq/timechart.csv",
                                trace_path="../raw/AMD_Ryzen5/chipseq/trace.csv",
                                output_path="../Processed_data/Experiments/AMD_Ryzen5/chipseq/measured/chipseq_ryzen5_measured.csv",
                                source_dataset="ryzen5_laptop", source_workflow="chipseq",
                                node="amd_ryzen_5", trace_sep = ",")


uProf covers workflow: True
Per-task energy attributed: 190 / 210 tasks (20 too short, fell between samples)
Total workflow energy: 24,730 J (6.87 Wh), mean 22.9 W over 18.1 min
  Attributed -> energy: 190, runtime: 210, memory: 210 (of 210)
  Sanity (rchar vs energy): Spearman 0.816 (n=190)
Saved: ../Processed_data/Experiments/AMD_Ryzen5/chipseq/measured/chipseq_ryzen5_measured.csv


In [ ]:
ryzen7_atacseq = process_uprof_run(timechart_path="../raw/AMD_Ryzen7/atacseq/timechart.csv",
                                trace_path="../raw/AMD_Ryzen7/atacseq/trace.csv",
                                output_path="../Processed_data/Experiments/AMD_Ryzen7/atacseq/measured/atacseq_ryzen7_measured.csv",
                                source_dataset="ryzen7_laptop", source_workflow="atacseq",
                                node="amd_ryzen_7", trace_sep = ",")

In [ ]:
ryzen7_chipseq = process_uprof_run(timechart_path="../raw/AMD_Ryzen7/chipseq/timechart.csv",
                                trace_path="../raw/AMD_Ryzen7/chipseq/trace.csv",
                                output_path="../Processed_data/Experiments/AMD_Ryzen7/chipseq/measured/chipseq_ryzen7_measured.csv",
                                source_dataset="ryzen7_laptop", source_workflow="chipseq",
                                node="amd_ryzen_7", trace_sep=",")

# Intel energy attribution

In [7]:
# Intel (RAPL): reconstruct per-interval energy from the cumulative counter
# (difference of consecutive readings, with wraparound correction), then sum
# the intervals falling inside each task's [start, complete] window.
def process_rapl_and_trace(rapl_path,trace_path,output_path,source_dataset,source_workflow,node,rapl_max_uj=262143328850
):
    # --- Load RAPL energy log ---
    rapl = pd.read_csv(rapl_path, header=None, names=["ts_ms", "energy_uj"])
    rapl = rapl.sort_values("ts_ms").reset_index(drop=True)

    # --- Reconstruct per-interval energy ---
    rapl["diff_uj"] = rapl["energy_uj"].diff()
    
    # Wraparound correction
    rapl.loc[rapl["diff_uj"] < 0, "diff_uj"] += rapl_max_uj
    
    # µJ -> J
    rapl["energy_j"] = rapl["diff_uj"] / 1e6
    rapl = rapl.dropna(subset=["energy_j"])
    rapl = rapl[rapl["energy_j"] >= 0]

    # --- Load trace ---
    tr = pd.read_csv(trace_path)
    tr = tr[tr["status"] == "COMPLETED"].copy()
    tr["start"] = pd.to_numeric(tr["start"], errors="coerce")
    tr["complete"] = pd.to_numeric(tr["complete"], errors="coerce")

    # --- Attribute energy: sum RAPL intervals within each task's window ---
    def task_energy(s, c):
        m = (rapl["ts_ms"] >= s) & (rapl["ts_ms"] <= c)
        return rapl.loc[m, "energy_j"].sum()

    tr["measured_energy_j"] = tr.apply(lambda r: task_energy(r["start"], r["complete"]), axis=1)

    # --- Add task_type and labels ---
    tr["task_type"] = tr["process"].str.split(":").str[-1].str.upper()
    tr["source_dataset"] = source_dataset
    tr["source_workflow"] = source_workflow
    tr["node"] = node

    # --- Keep tasks with real energy (drops sub-second tasks) ---
    tr = tr[tr["measured_energy_j"] > 0].copy()

    print(f"Tasks with measured energy: {len(tr)}")
    print(f"Total energy: {tr['measured_energy_j'].sum():.0f} J")
    print(f"Energy range: {tr['measured_energy_j'].min():.1f} to {tr['measured_energy_j'].max():.1f} J")

    # --- Save combined trace + energy ---
    tr.to_csv(output_path, index=False)
    print(f"Saved: {output_path}\n")

    return tr

In [8]:
# Intel i5: atacseq
df_atacseq = process_rapl_and_trace(
    rapl_path="../raw/Intel_i5/atacseq/timechart.csv",
    trace_path="../raw/Intel_i5/atacseq/trace.csv",
    output_path="../Processed_data/Experiments/intel_i5/atacseq/measured/intel_atacseq_measured.csv",
    source_dataset="intel_i5",
    source_workflow="atacseq",
    node="intel_i5"
)

# chipseq on the same Intel i5 machine
df_rnaseq = process_rapl_and_trace(
    rapl_path="../raw/Intel_i5/chipseq/timechart.csv",
        trace_path="../raw/Intel_i5/chipseq/trace.csv",
        output_path="../Processed_data/Experiments/intel_i5/chipseq/measured/intel_chipseq_measured.csv",
    source_dataset="intel_i5",
    source_workflow="chipseq",
    node="intel_i5"
)

Tasks with measured energy: 195
Total energy: 70810 J
Energy range: 5.8 to 4533.8 J
Saved: ../Processed_data/Experiments/intel_i5/atacseq/measured/intel_atacseq_measured.csv

Tasks with measured energy: 176
Total energy: 45246 J
Energy range: 5.4 to 3934.2 J
Saved: ../Processed_data/Experiments/intel_i5/chipseq/measured/intel_chipseq_measured.csv

